# 💳 Expense Tracker & Financial Behavior Analytics
### Industry-Grade Data Science Project
**Role Applicability:** Data Analyst | Business Analyst | Financial Analyst

---
## 📌 Project Objectives
1. **Data Ingestion & Cleaning:** Load raw transaction data, handle missing values, standardize text formats, and engineer temporal/financial features.
2. **Exploratory Data Analysis (EDA):** Analyze spending distributions across categories, payment channels, and calendar cycles.
3. **Financial Framework Auditing:** Benchmark actual spending behavior against the classic **50/30/20 Budgeting Rule** (Needs vs. Wants vs. Savings).
4. **Anomaly & Overspending Detection:** Identify high-risk deficit months, IQR transaction outliers, and spending velocity peaks.

In [ ]:
# Import core analytical libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

## 1️⃣ Load & Inspect Cleaned Financial Data

In [ ]:
# Load cleaned dataset
df = pd.read_csv("../data/cleaned_expenses.csv")
df["Date"] = pd.to_datetime(df["Date"])

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Schema and Data Types
df.info()

## 2️⃣ Executive Financial Metrics (KPIs)

In [ ]:
income = df[df["Type"] == "Income"]["Amount"].sum()
expense = df[df["Type"] == "Expense"]["Amount"].sum()
net_savings = income - expense
savings_rate = (net_savings / income) * 100 if income > 0 else 0

print(f"Total Inflow (Income):  ₹{income:,.2f}")
print(f"Total Outflow (Expense): ₹{expense:,.2f}")
print(f"Net Accumulated Savings: ₹{net_savings:,.2f}")
print(f"Savings Rate:            {savings_rate:.2f}%")

## 3️⃣ Category-Wise Spending Breakdown

In [ ]:
expense_df = df[df["Type"] == "Expense"]
cat_summary = expense_df.groupby("Category")["Amount"].agg(
    Total_Spend="sum",
    Txn_Count="count",
    Average_Txn="mean"
).reset_index()

cat_summary["Spend_Pct"] = (cat_summary["Total_Spend"] / expense * 100).round(2)
cat_summary = cat_summary.sort_values(by="Total_Spend", ascending=False)
cat_summary

In [ ]:
# Plot category distribution
plt.figure(figsize=(10, 5))
sns.barplot(data=cat_summary, x="Total_Spend", y="Category", palette="viridis")
plt.title("Total Expenditure per Category (₹)", fontsize=14, weight="bold")
plt.xlabel("Spend (₹)")
plt.show()

## 4️⃣ Monthly Cash Flow & Trend Analysis

In [ ]:
monthly = df.groupby(["Year_Month", "Type"])["Amount"].sum().unstack(fill_value=0).reset_index()
monthly["Net_Savings"] = monthly["Income"] - monthly["Expense"]

plt.figure(figsize=(12, 5))
plt.plot(monthly["Year_Month"], monthly["Income"], marker="o", label="Income", color="green", linewidth=2)
plt.plot(monthly["Year_Month"], monthly["Expense"], marker="s", label="Expense", color="red", linewidth=2)
plt.plot(monthly["Year_Month"], monthly["Net_Savings"], marker="^", label="Net Savings", color="blue", linestyle="--")
plt.xticks(rotation=45)
plt.title("Monthly Cash Inflow vs. Outflow Trajectory", fontsize=14, weight="bold")
plt.ylabel("Amount (₹)")
plt.legend()
plt.tight_layout()
plt.show()

## 5️⃣ 50/30/20 Budgeting Rule Audit

In [ ]:
needs = df[(df["Type"] == "Expense") & (df["Budget_Bucket"] == "Needs")]["Amount"].sum()
wants = df[(df["Type"] == "Expense") & (df["Budget_Bucket"] == "Wants")]["Amount"].sum()
savings = income - expense

actual = [round((needs / income) * 100, 1), round((wants / income) * 100, 1), round((savings / income) * 100, 1)]
target = [50.0, 30.0, 20.0]
labels = ["Needs", "Wants", "Savings"]

comp_df = pd.DataFrame({"Bucket": labels, "Target_Pct": target, "Actual_Pct": actual})
comp_df

## 6️⃣ Outlier Detection using IQR Method

In [ ]:
q1 = expense_df["Amount"].quantile(0.25)
q3 = expense_df["Amount"].quantile(0.75)
iqr = q3 - q1
threshold = q3 + (1.5 * iqr)

outliers = expense_df[expense_df["Amount"] > threshold]
print(f"Outlier Threshold: ₹{threshold:,.2f}")
print(f"Total Outlier Transactions Flagged: {len(outliers)}")
outliers[["Date", "Category", "Description", "Amount", "Payment_Method"]].head(10)